# Generate Manual Stylometry
### This file creates the style similarity created by teammate
##### Created by w45242hy

In [ ]:
!python -m spacy download en_core_web_md

In [ ]:
from pathlib import Path
root_path = Path().resolve().parent.parent
print(f"root_path: {root_path}")

In [ ]:
from transformers import BertModel, BertTokenizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
from spacy.symbols import ORTH, POS
from collections import defaultdict, Counter
import math
from nltk import ngrams
from sklearn.preprocessing import StandardScaler
import os

## Constants

In [ ]:
ROOT_PATH = os.path.dirname(os.path.dirname(os.getcwd()))

KAGGLE_DATASET_NAME = f"authorverification2"
KAGGLE_INPUT_PREFIX = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME)
TRIAL_DATA_PATH = f"{KAGGLE_INPUT_PREFIX}/AV_trial.csv"
TRAIN_DATA_PATH = f"{KAGGLE_INPUT_PREFIX}/train.csv"
VAL_DATA_PATH = f"{KAGGLE_INPUT_PREFIX}/dev.csv"
FASTTEXT_PATH = "/kaggle/input/datasets/yekenot/fasttext-crawl-300d-2m/crawl-300d-2M.vec"

nlp = spacy.load("en_core_web_md", disable=["ner"])

# ENCODER_MODEL_NAME = "bert-base-cased"

MAX_LEN = 256
BATCH_SIZE = 64            # For two GPUs (32 each)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data Loading

In [ ]:
trial_df = pd.read_csv(TRIAL_DATA_PATH, encoding="utf-8")
train_df = pd.read_csv(TRAIN_DATA_PATH, encoding="utf-8")
val_df = pd.read_csv(VAL_DATA_PATH, encoding="utf-8")

## Stylometry Feature Extraction

In [ ]:
# makes sure that these are lowercase
TOP_FUNCTION_WORDS = set(word.lower() for word in [
    "the", "of", "and", "to", "a", "in", 
    "is", "you", "that", "was", "he", "his", 
    "for", "it", "with", "as", "on", "are",
    "be", "at", "by", "they", "this", "from", 
    "i", "have", "or", "had", "not", "she"
])
SORTED_TOP_FUNCTION_WORDS = sorted(list(TOP_FUNCTION_WORDS))

FUNCTION_TAGS = {"ADP", "AUX", "CONJ", "CCONJ", "DET", "PART", "PRON", "SCONJ"}
CONTENT_TAGS = {"ADJ", "ADV", "INTJ", "NOUN", "NUM", "PROPN", "VERB"}

NON_WORD_TAGS = {"NUM", "PUNCT", "SYM", "X"}

ALL_NN_FORMS = {"NN", "NNS", "NNP", "NNPS"}                # all noun forms
ALL_JJ_FORMS = {"JJ", "JJR", "JJS"}                        # all adjective forms
ALL_RB_FORMS = {"RB", "RBR", "RBS"}                        # all adverb forms
ALL_VB_FORMS = {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"}   # all verb forms

def get_top_pos_trigrams():
    trigrams = set()

    for nn in ALL_NN_FORMS:
        trigrams.add((nn, "IN", "DT"))
        trigrams.add(("IN", "DT", nn))
        trigrams.add(("DT", nn, "IN"))
        trigrams.add(("DT", nn, "PUNCT"))
        for nn2 in ALL_NN_FORMS:
            trigrams.add(("DT", nn, nn2))

    for jj in ALL_JJ_FORMS:
        trigrams.add(("IN", "DT", jj))
        for nn in ALL_NN_FORMS:
            trigrams.add(("DT", jj, nn))
            trigrams.add((jj, nn, "IN"))
            trigrams.add((jj, nn, "PUNCT"))
            for rb in ALL_RB_FORMS:
                trigrams.add((rb, jj, nn))

    for vb in ALL_VB_FORMS:
        trigrams.add(("PRP", vb, "DT"))
        trigrams.add(("PRP", vb, "IN"))

    return trigrams
                
TOP_POS_TRIGRAMS = get_top_pos_trigrams()
SORTED_TOP_POS_TRIGRAMS = sorted(list(TOP_POS_TRIGRAMS))
# print(len(TOP_POS_TRIGRAMS))


def extract_stylometric_features(document):
    """
    Processes a spaCy document and returns a []-dimensional feature vector
    """
    f_count, c_count = 0, 0                          # counts of function and content words
    top_function_word_freqs = defaultdict(int)       # frequencies of top function words
    lexical_count = 0                                # count of all words
    unique_words = set()                             # set of all words (no duplicates)
    conjunction_count = 0                            # count of all conjunction words
    long_word_count = 0                              # long word (>6 chars) count
    oov_count, non_oov_count = 0, 0                  # counts of oov and vocabulary words
    word_length_sum = 0                              # sum of all word lengths
    pronoun_freqs = {                                # frequencies of various pronoun types
        "1S": 0,                             # first-person singular
        "1P": 0,                             # first-person plural
        "12": 0,                             # first- and second-person
        "3": 0                               # third-person
    }
    special_punct_freqs = defaultdict(int)           # frequencies of each of ";,.-?!"
    total_punct_count = 0                            # total punctuation count
    top_pos_trigram_freqs = defaultdict(int)         # frequencies of the top POS trigrams
    total_pos_trigram_count = 0                      # total POS trigram count

    # loop over all tokens
    for token in document:
        if token.is_space:
            continue

        text_lower = token.text.lower()
        universal_pos = token.pos_

        # word statistics
        if universal_pos not in NON_WORD_TAGS:
            unique_words.add(text_lower)
            lexical_count += 1
            word_length_sum += len(token.text)

            if text_lower in TOP_FUNCTION_WORDS:
                top_function_word_freqs[text_lower] += 1

            if universal_pos in FUNCTION_TAGS:
                f_count += 1
            elif universal_pos in CONTENT_TAGS:
                c_count += 1

            if universal_pos in ["CONJ", "CCONJ"]:
                conjunction_count += 1
            
            if len(token.text) > 6:
                long_word_count += 1

            if token.is_oov:
                oov_count += 1
            else:
                non_oov_count += 1

        # punctuation statistics
        if universal_pos == "PUNCT":
            total_punct_count += 1
            if token.text in ";,.-?!":
                special_punct_freqs[token.text] += 1

        # personal pronoun statistics
        if universal_pos == "PRON" and "Prs" in token.morph.get("PronType"):
            person = token.morph.get("Person")
            num = token.morph.get("Number")
            if "1" in person:
                pronoun_freqs["12"] += 1
                if "Sing" in num:
                    pronoun_freqs["1S"] += 1
                else:
                    pronoun_freqs["1P"] += 1
            elif "2" in person:
                pronoun_freqs["12"] += 1
            elif "3" in person:
                pronoun_freqs["3"] += 1

    # extract Penn Treebank POS tags
    ptb_tags = [token.tag_ for token in document]

    # loop over all POS trigrams
    for tri in ngrams(ptb_tags, 3):
        total_pos_trigram_count += 1
        if tri in TOP_POS_TRIGRAMS:
            top_pos_trigram_freqs[tri] += 1

    # sentence statistics
    sentence_lengths = [len([t for t in s if not t.is_space]) for s in document.sents]
    mean_sentence_length = np.mean(sentence_lengths) if sentence_lengths else 0
    if len(sentence_lengths) > 1:
        variance = sum((length - mean_sentence_length)**2 for length in sentence_lengths) / len(sentence_lengths)
        std_sentence_length = math.sqrt(variance)
    else:
        std_sentence_length = 0

    # final feature assembly
    feature_vector = []

    # the (normalised) frequencies of each of the top 30 function words [30 features]
    for word in SORTED_TOP_FUNCTION_WORDS:
        feature_vector.append(top_function_word_freqs[word] / (f_count + 1e-9))

    # the (normalised) ratio of function words to content words [1 feature]
    feature_vector.append(np.clip(f_count / (c_count + 1e-9), 0, 5))

    # the (normalised) type-token ratio (i.e proportion of non-unique words) [1 feature]
    feature_vector.append(len(unique_words) / (lexical_count + 1e-9))

    # the (normalised) count of conjunctions (e.g. and, but) [1 feature]
    feature_vector.append(conjunction_count / (f_count + 1e-9))

    # the (normalised) count of long words (>6 chars) [1 feature]
    feature_vector.append(long_word_count / (lexical_count + 1e-9))

    # the (normalised) ratio of OOV words to non-OOV words [1 feature]
    feature_vector.append(np.clip(oov_count / (non_oov_count + 1e-9), 0, 5))

    # the (normalised) ratio of first- and second-person pronouns to third-person pronouns [1 feature]
    feature_vector.append(np.clip(pronoun_freqs["12"] / (pronoun_freqs["3"] + 1e-9), 0, 5))

    # the (normalised) ratio of first-person singular pronouns to first-person plural pronouns [1 feature]
    feature_vector.append(np.clip(pronoun_freqs["1S"] / (pronoun_freqs["1P"] + 1e-9), 0, 5))

    # the mean word length [1 feature]
    feature_vector.append(np.clip(word_length_sum / (lexical_count + 1e-9), 0, 20))

    # the mean sentence length [1 feature]
    feature_vector.append(np.clip(mean_sentence_length, 0, 100))

    # the sentence length standard deviation [1 feature]
    feature_vector.append(np.clip(std_sentence_length, 0, 100))

    # the (normalised) frequencies of punctuation symbols in ';,.-?!' [6 features]
    for symbol in ";,.-?!":
        feature_vector.append(special_punct_freqs[symbol] / (total_punct_count + 1e-9))

    # the (normalised) ratios of punctuation symbol pairs of interest [6 features]
    punctuation_pairs = [
        (",", ";"), (",", "-"),
        (";", "-"), ("?", "."),
        (",", "."), ("!", ".")
    ]
    for s1, s2 in punctuation_pairs:
        feature_vector.append(np.clip(special_punct_freqs[s1] / (special_punct_freqs[s2] + 1e-9), 0, 5))

    # the (normalised) counts of the top POS trigrams [119 features]
    for tri in SORTED_TOP_POS_TRIGRAMS:
        feature_vector.append(top_pos_trigram_freqs[tri] / (total_pos_trigram_count + 1e-9))

    return np.array(feature_vector)

def process_input_stylometry(df):
    feature_vectors_1 = []
    feature_vectors_2 = []

    for document in nlp.pipe(df["text_1"], batch_size=256, n_process=-1, disable=["ner", "lemmatizer"]):
        feature_vectors_1.append(extract_stylometric_features(document))

    for document in nlp.pipe(df["text_2"], batch_size=256, n_process=-1, disable=["ner", "lemmatizer"]):
        feature_vectors_2.append(extract_stylometric_features(document))
        
    return np.array(feature_vectors_1), np.array(feature_vectors_2)

In [ ]:
train_style_1, train_style_2 = process_input_stylometry(train_df)
val_style_1, val_style_2 = process_input_stylometry(val_df)

In [ ]:
# Different features can have different ranges, so we need to scale them

combined_train_style = np.vstack([train_style_1, train_style_2])
combined_val_style = np.vstack([val_style_1, val_style_2])

scaler = StandardScaler().fit(combined_train_style)

# Scale all features
train_style_scaled_1 = scaler.transform(train_style_1)
train_style_scaled_2 = scaler.transform(train_style_2)

val_style_scaled_1 = scaler.transform(val_style_1)
val_style_scaled_2 = scaler.transform(val_style_2)

# Compute the difference
train_style_diff = np.abs(train_style_scaled_1 - train_style_scaled_2)
val_style_diff = np.abs(val_style_scaled_1 - val_style_scaled_2)

print(f"train_style_diff: {train_style_diff.shape}")
print(f"val_style_diff: {val_style_diff.shape}")

train_df = pd.DataFrame(train_style_diff)
val_df = pd.DataFrame(val_style_diff)

train_df.to_csv("train_style_diff.csv", index=False)
val_df.to_csv("val_style_diff.csv", index=False)